> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

> The original notebook also contained a cell that generated a synthetic 'accuracy 33% -> 87%' curve with `random.random()`. It has been removed; only the measured loss-curve cell remains.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import torch.optim as optim
from glob import glob
import os
import mne
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm
import torch._dynamo


torch._dynamo.config.suppress_errors = True

import sys
import os
from pathlib import Path
# Get the current working directory
notebook_path = Path().absolute()
# Go up one directory level to reach myproject
project_root = notebook_path.parent
# Add to Python path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
# Now you can import
from utils import read_eeg, balance_labels, make_features

In [ ]:
print(torch.cuda.is_available())

# preprocess

In [ ]:
from sklearn.model_selection import train_test_split
X, y = read_eeg(win_size=0.5, why="test", verbose=False, overlap=0.4)


indexes0 = np.where(y == 0)[0]
indexes1 = np.where(y == 1)[0]
indexes2 = np.where(y == 2)[0]

zeros = np.unique(y, return_counts=True)[1][0]
ones = np.unique(y, return_counts=True)[1][1]
twos = np.unique(y, return_counts=True)[1][2]

MIN = min(zeros, ones, twos)

random_indexes0 = np.random.choice(indexes0, size=zeros-MIN if zeros-MIN > 0 else 0, replace=False)
random_indexes1 = np.random.choice(indexes1, size=ones-MIN if ones-MIN > 0 else 0, replace=False)
random_indexes2 = np.random.choice(indexes2, size=twos-MIN if twos-MIN > 0 else 0, replace=False)
# print(random_indexes)
y = np.delete(y, np.concatenate((random_indexes0, random_indexes1, random_indexes2)))
X = np.delete(X, np.concatenate((random_indexes0, random_indexes1, random_indexes2)), axis=0)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
print(X.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=30, Samples=500, dropoutRate=0.5, 
                 kernLength=64, F1=16, D=2, F2=32, norm_rate=0.1):
        super(EEGNet, self).__init__()
        
        # Temporal convolution
        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False),
            nn.BatchNorm2d(F1, momentum=norm_rate),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.3)  # Added dropout after first layer
        )
        
        # Depthwise convolution with residual
        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1*D, momentum=norm_rate),
            nn.LeakyReLU(0.1),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropoutRate)
        )
        
        # Separable convolution with residual
        self.separable = nn.Sequential(
            nn.Conv2d(F1*D, F1*D, (1, 16), padding='same', groups=F1*D, bias=False),
            nn.Conv2d(F1*D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2, momentum=norm_rate),
            nn.LeakyReLU(0.1),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropoutRate)
        )
        
        self.flatten_size = self._calculate_flatten_size(Chans, Samples)
        self.classifier = nn.Sequential(
            nn.Linear(self.flatten_size, 128),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.5),
            nn.Linear(128, nb_classes)
        )

    def _calculate_flatten_size(self, Chans, Samples):
        with torch.no_grad():
            x = torch.zeros(1, 1, Chans, Samples)
            x = self.temporal(x)
            x = self.spatial(x)
            x = self.separable(x)
            return int(np.prod(x.size()[1:]))

    def forward(self, x):
        # Input shape: (batch, 1, channels, time_samples)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

In [ ]:
print(np.unique(y, return_counts=True))

In [ ]:
# (10224, 500, 30)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[2], X_train.shape[1], 1)  # Reshape to (..., 30, 250, 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[2], X_test.shape[1], 1)  # Reshape to (..., 30, 250, 1)

print(X_train.shape)  # should be (samples, 30, 250, 1)
print(y_train.shape)  # should be (samples, 3)


In [ ]:
from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=3) 
y_test = to_categorical(y_test, num_classes=3) 

print(f"Y train\n{y_train}")


print("X Train: " + str(X_train.shape))
print("Y Train: " + str(y_train.shape))

In [ ]:
X_train = torch.tensor(X_train).permute(0, 3, 1, 2).float()
y_train = torch.tensor(y_train).float()

X_test = torch.tensor(X_test).permute(0, 3, 1, 2).float()
y_test = torch.tensor(y_test).float()

X.shape

In [ ]:
y_train = torch.argmax(y_train, dim=1)
y_test = torch.argmax(y_test, dim=1)


In [ ]:
n_chan = 30
print(n_chan)
print(y_train.shape)

In [ ]:
# MY GRIDSEARCH
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display, clear_output, Markdown
import datetime
from sklearn.metrics import confusion_matrix
# Create datasets and loaders

# 64 0.001 0.4 32 16 4 32

batches = [128, 64] # [8, 16, 32, 64, 128] 32
learnings = [ 0.001, 0.005, 0.01] #  [0.001, 0.01] 0.01
drops = [0.2, 0.3, 0.4, 0.5] # 0.4
kernLengths = [32, 64] # 64
F1 = [16] # [64, 32, 16] 64
D = [2]
F2 = [16] # [64, 32, 16] 64
norm_rate = [0.4]

output = ""
MAX_TRIES = 15

accuracies = []
losses_t = []

best_best = 0

models = []
for batch in batches:
    for learning in learnings:
        for drop in drops:
            for kernLength in kernLengths:
                for f1 in F1:
                    for d in D :
                        for f2 in F2:
                            for norm in norm_rate:
                                # print(f"Batch: {batch}, Learning Rate: {learning}, Dropout: {drop}, Kernel Length: {kernLength}, F1: {f1}, D: {d}, F2: {f2}")
                                output += f"**Batch:** {batch}, **Learning Rate:** {learning}, **Dropout:** {drop}, **Kernel Length:** {kernLength}, **F1:** {f1}, **D:** {d}, **F2:** {f2}\n\n"

                                train_dataset = TensorDataset(X_train , y_train )
                                train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=False)
                                test_dataset = TensorDataset(X_test, y_test)
                                test_loader = DataLoader(test_dataset, batch_size=batch, shuffle=True)
                                #model = EEGNet(nb_classes=3, Chans=30, Samples=250, dropoutRate=drop, kernLength=kernLength)
                                model = EEGNet(nb_classes=3, Chans=n_chan, Samples=250, dropoutRate=drop, kernLength=kernLength,
                                            F1=f1, D=d, F2=f2, norm_rate=norm)
                                device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                                # print(device)
                                model = model.to(device)

                                criterion = torch.nn.CrossEntropyLoss()
                                optimizer = torch.optim.AdamW(model.parameters(), lr=learning, weight_decay=1e-4)

                                epochs = 400
                                

                                best_model = (None, 0, 0)  # (model, accuracy, epock)
                                losses = 5
                                tries = MAX_TRIES
                                best_val = 0

                                for epoch in range(epochs):
                                    correct = 0
                                    total = 0
                                    model.train()
                                    total_loss = 0
                                    start = datetime.datetime.now()
                                    for X_batch, y_batch in train_loader:
                                        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                                        

                                        optimizer.zero_grad()
                                        outputs = model(X_batch)
                                        loss = criterion(outputs, y_batch)
                                        loss.backward()
                                        optimizer.step()
                                        total_loss += loss.item()
                                        pred = outputs.argmax(dim=1)
                                        correct += (pred == y_batch).sum().item()
                                        total += y_batch.size(0)
                                    accuracy = 100 * correct / total
                                    end = datetime.datetime.now()
                                    # Disable gradient calculation for evaluation
                                    model.eval()
                                    correct = 0
                                    total = 0
                                    with torch.no_grad():
                                        cn = 0
                                        cp = 0
                                        cw = 0
                                        for X_batch, y_batch in train_loader:
                                            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                                            outputs = model(X_batch)
                                    
                                            _, predicted = torch.max(outputs.data, 1)
                                            total += y_batch.size(0)
                                            correct += (predicted == y_batch).sum().item()
            
                                            for p in predicted:
                                                if p == 0:
                                                    cn += 1
                                                elif p == 1:
                                                    cp += 1
                                                else:
                                                    cw += 1
                                            
                                        accuracy = 100 * correct / total
                                        best_t_a = 0
                                        val_correct = 0
                                        val_total = 0
                                        for X_batch, y_batch in test_loader:
                                            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                                            outputs = model(X_batch)
                                            _, predicted = torch.max(outputs.data, 1)
                                            val_total += y_batch.size(0)
                                            val_correct += (predicted == y_batch).sum().item()
                                        val_accuracy = 100 * val_correct / val_total
                                                
                                    best = False
                                    if val_accuracy > best_val:
                                        best_val = val_accuracy
                                        best_model = (model, accuracy, epoch)
                                        best = True
                                        if accuracy > best_best:
                                            best_best = accuracy
                                            
                                    LS = total_loss / len(train_loader)

                                    accuracies.append(val_accuracy)
                                    losses_t.append(LS)
                                    display(Markdown(output + f"- **Epoch {epoch+1}/{epochs}**, **Loss:** {LS:.4f}, **Acc:** {accuracy:.2f}%, **Val Acc:** {val_accuracy:.2f}%  - ({((end - start).total_seconds()):.2f}s)  **ES:** *{tries}/{MAX_TRIES}*."), clear=True)
                                    if accuracy == best_best or best:
                                        output += f"- **Epoch {epoch+1}/{epochs}**, **Loss:** {LS:.4f}, **Acc:** __{accuracy:.2f}__%, **Val Acc:** __{val_accuracy:.2f}__%  - ({((end - start).total_seconds()):.2f}s)  **ES:** *{tries}/{MAX_TRIES}*.  !\n\n"

                                    # print(f"- Epoch {epoch+1}/{epochs}, Loss: {LS:.4f}, Acc: {accuracy:.4f}%, Val Acc: {val_accuracy:.4f}%  - Time: {((end - start).total_seconds()):.2f}s {" -  !! Best !!" if best else ""}")
                                    if LS <= losses or best:
                                        losses = LS
                                        tries = MAX_TRIES
                                    else:
                                        tries -= 1
                                    if tries == 0:
                                        break
                                # if accuracy != best_best:
                                #     output += f"- **Epoch {epoch+1}/{epochs}**, **Loss:** {LS:.4f}, **Acc:** __{accuracy:.2f}__%, **Val Acc:** __{val_accuracy:.2f}__%  - ({((end - start).total_seconds()):.2f}s)  **ES:** *{tries}/{MAX_TRIES}*.\n\n"


                                models.append(( best_model, batch, learning, drop, kernLength, f1, d, f2))


In [ ]:
# plot data accuracies / epochs
plt.figure(figsize=(12, 6))
# plt.plot(accuracies[:140], label='Validation Accuracy', color='blue')

plt.plot(losses_t[:140], label='Validation lOSS', color='blue')
plt.title('Validation lOSS Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('lOSS')
plt.legend()
plt.grid()
plt.show()

# test

In [ ]:
best_model